# **Introduction**

This assignment explores the full spectrum of supervised‐learning techniques—classification, regression, and ensemble methods—through a series of practical exercises grounded in real‐world scenarios. I begin by framing a predictive‐analytics solution for reducing 30-day readmissions among diabetes patients in a large hospital group, then hand-compute decision-tree splits (ID3), nearest-neighbor distances, and Naïve Bayes probabilities on toy datasets. Next, I derive and fit linear and logistic regression models (including gradient-descent updates), evaluate a support-vector machine’s decision function on given support vectors, and finally combine multiple classifiers using both bagging and boosting approaches. Throughout, I interpret results in business terms, critique model assumptions, and suggest data-driven improvements—all in service of mastering the end-to-end process of building and evaluating supervised-learning solutions.

---

**Learning Objectives**

By completing this assignment, I will:

1. **Design predictive-analytics solutions**
   Propose and justify two AI-driven approaches to lower hospital readmission rates without extending stays (classification models, care-plan recommendations).

2. **Build and interpret decision trees (ID3)**
   Compute entropies and information gains; select root features; critique dataset designs that lack contrasting indicators.

3. **Apply k-Nearest Neighbors**
   Calculate overlap (Hamming) and Euclidean distances; make 1-NN and k-NN predictions; discuss the impact of the choice of $k$.

4. **Implement Naïve Bayes classification**
   Estimate prior and conditional probabilities from data; classify new instances under the conditional-independence assumption.

5. **Perform linear regression analysis**
   Write the squared-error loss; predict with given weights; compute total SSE; execute one gradient-descent update; re-evaluate SSE.

6. **Execute logistic regression modeling**
   Derive the log-odds link and sigmoid form; compute predictions and loss; suggest model improvements for poor fit.

7. **Evaluate Support Vector Machines**
   Compute decision-function values for query points given support vectors, Lagrange multipliers, and bias.

8. **Construct ensemble classifiers**
   Combine multiple base models via majority voting (bagging) and weighted voting (boosting); compute ensemble outputs and misclassification rates.

9. **Communicate findings**
   Translate technical results into actionable business and clinical insights; reflect on model limitations; propose next steps.


## **Question 1**
The management of a large hospital group are concerned about the high 30-day readmission rate among diabetes patients (≈20 % vs. 11 % overall). They suspect premature discharge or incomplete in-hospital care plans may be to blame. Hospital management wants to leverage AI to reduce readmissions without unnecessarily extending stays.

---

**i) Two Predictive-Analytics Solutions**

1. **Discharge-Risk Classifier**

   * **Model:** Train a binary classifier (e.g. logistic regression, random forest) on historical admissions to predict “readmit within 30 days” vs. “no readmission.”
   * **Usage:** At discharge, score each patient’s risk. For those above a threshold, trigger enhanced interventions—extended observation, medication reviews, or home visits—rather than automatic discharge.
   * **Impact:** By focusing resources on high-risk patients, the hospital can reduce avoidable readmissions (and associated penalties) while preserving overall throughput.

2. **Care-Plan Personalization Engine**

   * **Model:** Build a multi-label recommender (e.g. based on collaborative filtering or decision-tree ensembles) that suggests tailored care-plan elements—diet counseling, glucose monitoring frequency, referrals to specialists—based on similarities to past patients with low readmission rates.
   * **Usage:** During the hospital stay, generate a customized bundle of follow-up services aligned to each patient’s risk profile. Clinicians review and implement the top recommended actions.
   * **Impact:** Personalized care plans address individual needs that generic protocols might miss, lowering the chance of complications and readmissions without blanket longer stays.

---

**ii) Data Requirements for the Discharge-Risk Classifier**

To train a readmission-risk model, we need:

* **Demographics:** age, sex, ethnicity
* **Clinical history:** diabetes type, comorbidities (e.g. hypertension, renal disease), prior admissions
* **In-hospital metrics:** lab values (HbA1c, blood glucose), vital signs, procedures performed, medication regimens
* **Length of stay** and **discharge disposition** (e.g. home, rehab)
* **Post-discharge outcomes:** whether readmitted within 30 days, reason for readmission

This structured data lets the classifier learn which combinations of features most strongly predict short-term readmissions .

---

**iii) Hospital Capacity & Infrastructure Needs**

To operationalize this solution, the hospital must invest in:

1. **Data infrastructure:**

   * A centralized EHR warehouse integrating demographics, labs, medications, and discharge notes.
   * A real-time scoring service to compute risk scores as patients approach discharge.

2. **Analytics team & tooling:**

   * Data engineers to maintain pipelines and ensure data quality.
   * Data scientists/ML engineers to retrain models periodically and monitor performance.

3. **Care-management resources:**

   * Dedicated staff (nurses, pharmacists, case managers) to receive and act on high-risk alerts.
   * Capacity for follow-up care (e.g. scheduling telehealth visits, home nursing) to support flagged patients without creating new bottlenecks.

With these capabilities, the hospital can harness predictive analytics to reduce readmissions while maintaining efficient patient flow.


## **Question 2**
> We have six patients, each described by three binary features and a target:
>
> | ID | OBESE | SMOKER | DRINKS\_ALCOHOL | CANCER\_RISK |
> | -- | :---: | :----: | :-------------: | :----------: |
> | 1  |  true |  false |       true      |      low     |
> | 2  |  true |  true  |       true      |     high     |
> | 3  |  true |  false |       true      |      low     |
> | 4  | false |  true  |       true      |     high     |
> | 5  | false |  true  |      false      |      low     |
> | 6  | false |  true  |       true      |     high     |

### (a) Which feature is chosen at the root by ID3?

We first compute the **base entropy** of the target,

$$
H(\text{Cancer\_Risk})
= -\sum_{v\in\{\text{low},\text{high}\}}
  p(v)\,\log_{2} p(v)
= -\Bigl(\tfrac{3}{6}\log_{2}\tfrac{3}{6} \;+\; \tfrac{3}{6}\log_{2}\tfrac{3}{6}\Bigr)
= 1.0\text{ bit.}
$$

Then for each candidate feature $X$, we compute the entropy of the target **conditional** on $X$, and the resulting **information gain**:

$$
IG(X)
= H(\text{Cancer\_Risk})
  \;-\;
  \sum_{x\in\{\mathit{true},\mathit{false}\}}
    \Pr(X=x)\;
    H\bigl(\text{Cancer\_Risk}\mid X=x\bigr).
$$

---

#### 1. OBESE

* **When OBESE = true** (IDs 1,2,3):
  target = $\{\text{low},\text{high},\text{low}\}$ →
  $H = -\bigl(\tfrac{2}{3}\log_2\tfrac{2}{3} + \tfrac{1}{3}\log_2\tfrac{1}{3}\bigr)\approx0.9183.$

* **When OBESE = false** (IDs 4,5,6):
  target = $\{\text{high},\text{low},\text{high}\}$ →
  $H\approx0.9183$ as well.

* Weighted average conditional entropy

  $$
    \tfrac{3}{6}\cdot0.9183 + \tfrac{3}{6}\cdot0.9183 = 0.9183.
  $$

* **Information gain**

  $$
    IG(\mathrm{OBESE})
    = 1.0 - 0.9183
    = 0.0817\text{ bits.}
  $$

---

#### 2. SMOKER

* **When SMOKER = true** (IDs 2,4,5,6):
  target = $\{\text{high},\text{high},\text{low},\text{high}\}$ →

  $$
    H = -\bigl(\tfrac{3}{4}\log_2\tfrac{3}{4} + \tfrac{1}{4}\log_2\tfrac{1}{4}\bigr)
      \approx0.8113.
  $$

* **When SMOKER = false** (IDs 1,3):
  target = $\{\text{low},\text{low}\}$ →
  $H = -\bigl(1\cdot\log_2 1 + 0\bigr) = 0.$

* Weighted average

  $$
    \tfrac{4}{6}\cdot0.8113 + \tfrac{2}{6}\cdot0
    = 0.541.
  $$

* **Information gain**

  $$
    IG(\mathrm{SMOKER})
    = 1.0 - 0.541
    = 0.459\text{ bits.}
  $$

---

#### 3. DRINKS\_ALCOHOL

* **When DRINKS\_ALCOHOL = true** (IDs 1,2,3,4,6):
  target = $\{\text{low},\text{high},\text{low},\text{high},\text{high}\}$ →

  $$
    H
    = -\Bigl(\tfrac{2}{5}\log_2\tfrac{2}{5}
           + \tfrac{3}{5}\log_2\tfrac{3}{5}\Bigr)
    \approx0.9709.
  $$

* **When DRINKS\_ALCOHOL = false** (ID 5):
  target = $\{\text{low}\}$ → $H=0.$

* Weighted average

  $$
    \tfrac{5}{6}\cdot0.9709 + \tfrac{1}{6}\cdot0
    = 0.809.
  $$

* **Information gain**

  $$
    IG(\mathrm{DRINKS\_ALCOHOL})
    = 1.0 - 0.809
    = 0.191\text{ bits.}
  $$

---

### Root‐feature selection

Since **SMOKER** has the **highest** information gain (≈ 0.459 bits), ID3 will choose **SMOKER** as the root node.

---

### (b) Suggested features that indicate **low** cancer risk

A well‐balanced dataset should include some descriptive features that, when **true**, correlate with **low** risk. Examples we could add:

* **REGULAR\_EXERCISE**: true if the patient exercises ≥ 150 minutes per week.
* **GOOD\_GLYCEMIC\_CONTROL**: true if the patient’s HbA₁c is consistently < 7 %.
* **HEALTHY\_DIET**: true if the patient follows a balanced, low‐sugar nutrition plan.

Each of these, when true, would tend to appear only in **low**–risk patients, providing informative, contrasting splits in the tree.


## **Question 3**

To decide the best root split, I compare the total Sum­ of­ Squared Errors (SSE) for each candidate feature (STUDIED vs. ENERGY).  The SSE for a split on feature $X$ is

$$
\mathrm{SSE}(X)
=\sum_{x\in\{\mathit{yes},\mathit{no}\}}
\sum_{i: X_i = x}\bigl(y_i - \bar y_{x}\bigr)^2,
$$

where $\bar y_{x}$ is the mean score among instances with $X=x$.

The data (from the assignment PDF) is:

| ID | STUDIED | ENERGY | SCORE |   |
| -- | :-----: | :----: | ----: | - |
| 1  |   yes   |  tired |    65 |   |
| 2  |    no   |  alert |    20 |   |
| 3  |   yes   |  alert |    90 |   |
| 4  |   yes   |  tired |    70 |   |
| 5  |    no   |  tired |    40 |   |
| 6  |   yes   |  alert |    85 |   |
| 7  |    no   |  tired |    35 |   |

---

1. **Split on STUDIED**

* **Group “yes”** ($i$=1,3,4,6): scores $\{65,90,70,85\}$

  $$
    \bar y_{\text{yes}}
    = \frac{65+90+70+85}{4}
    = 77.5,
  $$

  $$
    \mathrm{SSE}_{\text{yes}}
    = (65-77.5)^2 + (90-77.5)^2 + (70-77.5)^2 + (85-77.5)^2
    = 156.25 + 156.25 + 56.25 + 56.25
    = 425.
  $$

* **Group “no”** ($i$=2,5,7): scores $\{20,40,35\}$

  $$
    \bar y_{\text{no}}
    = \frac{20+40+35}{3}
    \approx 31.67,
  $$

  $$
    \mathrm{SSE}_{\text{no}}
    = (20-31.67)^2 + (40-31.67)^2 + (35-31.67)^2
    \approx 136.11 + 69.44 + 11.11
    = 216.67.
  $$

* **Total SSE**

  $$
    \mathrm{SSE}(\text{STUDIED})
    = 425 + 216.67
    = 641.67.
  $$

---

2. **Split on ENERGY**

* **Group “alert”** ($i$=2,3,6): scores $\{20,90,85\}$

  $$
    \bar y_{\text{alert}}
    = \frac{20+90+85}{3}
    = 65,
  $$

  $$
    \mathrm{SSE}_{\text{alert}}
    = (20-65)^2 + (90-65)^2 + (85-65)^2
    = 2025 + 625 + 400
    = 3050.
  $$

* **Group “tired”** ($i$=1,4,5,7): scores $\{65,70,40,35\}$

  $$
    \bar y_{\text{tired}}
    = \frac{65+70+40+35}{4}
    = 52.5,
  $$

  $$
    \mathrm{SSE}_{\text{tired}}
    = (65-52.5)^2 + (70-52.5)^2 + (40-52.5)^2 + (35-52.5)^2
    = 156.25 + 306.25 + 156.25 + 306.25
    = 925.
  $$

* **Total SSE**

  $$
    \mathrm{SSE}(\text{ENERGY})
    = 3050 + 925
    = 3975.
  $$

---

3. **Decision**

Because

$$
\mathrm{SSE}(\text{STUDIED}) = 641.67
\;<\;
\mathrm{SSE}(\text{ENERGY}) = 3975,
$$

splitting on **STUDIED** yields a much lower error.  Therefore, I would use **STUDIED** as the root‐node criterion.


## **Question 4: 1-NN and k-NN on the “Galápagos” animals** 

We have 8 binary features.  For each animal (IDs 1–4) and the mystery “Query,” I’ll write them in this fixed order:

1. **Births live young**
2. **Lays eggs**
3. **Feeds offspring own milk**
4. **Warm-blooded**
5. **Cold-blooded**
6. **Land & water based**
7. **Has hair**
8. **Has feathers**

|     ID    |  1  |  2  |  3  |  4  |  5  |  6  |  7  |  8  | Class     |
| :-------: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-------- |
| **Query** |  F  |  T  |  F  |  F  |  F  |  T  |  F  |  F  | ?         |
|   **1**   |  T  |  F  |  T  |  T  |  F  |  F  |  T  |  F  | mammal    |
|   **2**   |  F  |  T  |  F  |  F  |  T  |  T  |  F  |  F  | amphibian |
|   **3**   |  T  |  F  |  T  |  T  |  F  |  F  |  T  |  F  | mammal    |
|   **4**   |  F  |  T  |  F  |  T  |  F  |  T  |  F  |  T  | bird      |

### a) Compute Hamming (overlap) distances

The Hamming distance between two binary vectors is simply the count of positions where they differ.  I’ll compare each ID’s row to the Query row (F T F F F T F F):

* **ID 1 vs Query**

  ```
  Query:  F  T  F  F  F  T  F  F  
  ID1:    T  F  T  T  F  F  T  F  
           ↑  ↑  ↑  ↑      ↑     mismatches
  ```

  Number of mismatches = **6**.

* **ID 2 vs Query**

  ```
  Query:  F  T  F  F  F  T  F  F  
  ID2:    F  T  F  F  T  T  F  F  
                ↑               one mismatch
  ```

  Number of mismatches = **1**.

* **ID 3 vs Query**

  ```
  Query:  F  T  F  F  F  T  F  F  
  ID3:    T  F  T  T  F  F  T  F  
           ↑  ↑  ↑  ↑      ↑     mismatches
  ```

  Number of mismatches = **6**.

* **ID 4 vs Query**

  ```
  Query:  F  T  F  F  F  T  F  F  
  ID4:    F  T  F  T  F  T  F  T  
                ↑           ↑   two mismatches
  ```

  Number of mismatches = **2**.

Putting it all together:

|  ID | Distance to Query |
| :-: | :---------------: |
|  1  |         6         |
|  2  |         1         |
|  3  |         6         |
|  4  |         2         |

### b) 1-Nearest-Neighbor classification

With $k=1$, I pick the single closest animal—ID 2 (distance = 1).  Since ID 2’s class is **amphibian**, I assign the mystery animal to the **amphibian** class.

### c) 4-Nearest-Neighbors classification

For $k=4$, I take the four smallest distances: IDs 2, 4, 1, 3 (distances 1, 2, 6, 6).  Their classes are:

* ID 2 → amphibian
* ID 4 → bird
* ID 1 → mammal
* ID 3 → mammal

That gives a vote tally:

* **mammal** = 2
* amphibian = 1
* bird = 1

So the majority class is **mammal**, and I would (perhaps surprisingly) label the mystery animal as a mammal under 4-NN.

#### Is $k=4$ a good choice?

Not really—two of the “four” nearest (IDs 1 & 3) actually live in a very different part of feature‐space (distance = 6), and the vote comes down to “farther” neighbors rather than the ones most similar to Query.  Moreover, an even $k$ risks ties; here it luckily broke 2–1–1, but had I chosen ID 4 over ID 1 we could get a 2–2 split.

In practice I’d prefer an **odd** $k$ (e.g.\ 1 or 3) so ties can’t happen, and I’d keep $k$ small enough that only genuinely close neighbors influence the decision.


## **Question 5: Naïve Bayes on Book‐Purchase Data**

We have 10 examples, each with

* target $Y=\text{PURCHASED}\in\{\mathsf{true},\mathsf{false}\}$,
* two binary/categorical features:

  * **SECONDHAND** $\in\{\mathsf{true},\mathsf{false}\}$,
  * **GENRE** $\in\{\mathsf{romance},\mathsf{science},\mathsf{literature}\}$,
  * **COST** $\in\{\mathsf{cheap},\mathsf{reasonable},\mathsf{expensive}\}$.

|  ID | SECONDHAND |    GENRE   |    COST    | PURCHASED |
| :-: | :--------: | :--------: | :--------: | :-------: |
|  1  |    false   |   romance  |  expensive |    true   |
|  2  |    false   |   science  |    cheap   |   false   |
|  3  |    true    |   romance  |    cheap   |    true   |
|  4  |    false   |   science  |    cheap   |    true   |
|  5  |    false   |   science  |  expensive |   false   |
|  6  |    true    |   romance  | reasonable |   false   |
|  7  |    true    | literature |    cheap   |   false   |
|  8  |    false   |   romance  | reasonable |   false   |
|  9  |    true    |   science  |    cheap   |   false   |
|  10 |    true    | literature | reasonable |    true   |

### a) Estimate the Naïve‐Bayes probabilities

#### 1) Priors

$$
P(Y=\mathsf{true})=\frac{\#\{\text{true}\}}{10}=\frac{4}{10}=0.4000,\quad
P(Y=\mathsf{false})=\frac{6}{10}=0.6000.
$$

#### 2) Conditionals

We count separately over the 4 “purchased = true” cases and the 6 “false” cases.

|     Feature    |    Value   | Count $\mid Y=\mathsf{true}$ | Prob $\,P(\cdot\mid Y=\mathsf{true})$ | Count $\mid Y=\mathsf{false}$ | Prob $\,P(\cdot\mid Y=\mathsf{false})$ |
| :------------: | :--------: | :--------------------------: | :-----------------------------------: | :---------------------------: | :------------------------------------: |
| **SECONDHAND** |    true    |               2              |              $2/4=0.5000$             |               3               |              $3/6=0.5000$              |
|                |    false   |               2              |              $2/4=0.5000$             |               3               |              $3/6=0.5000$              |
|    **GENRE**   |   romance  |               2              |              $2/4=0.5000$             |               2               |           $2/6\approx0.3333$           |
|                |   science  |               1              |              $1/4=0.2500$             |               3               |              $3/6=0.5000$              |
|                | literature |               1              |              $1/4=0.2500$             |               1               |           $1/6\approx0.1667$           |
|    **COST**    |    cheap   |               2              |              $2/4=0.5000$             |               3               |              $3/6=0.5000$              |
|                | reasonable |               1              |              $1/4=0.2500$             |               2               |           $2/6\approx0.3333$           |
|                |  expensive |               1              |              $1/4=0.2500$             |               1               |           $1/6\approx0.1667$           |

---

### b) Class‐posterior for a new book

**Query**:

$$
\text{SECONDHAND}=\mathsf{false},\quad 
\text{GENRE}=\text{literature},\quad
\text{COST}=\text{expensive}.
$$

By the Naïve‐Bayes assumption,

$$
P(Y=y\mid \mathbf{x})
\;\propto\; P(Y=y)\,\prod_i P(x_i\mid Y=y).
$$

Compute un‐normalized scores:

$$
\begin{aligned}
\tilde P(Y=\mathsf{true},\mathbf{x})
&=0.4000
\;\times\;P(\mathsf{false}\mid\mathsf{true})\,
            P(\text{lit}\mid\mathsf{true})\,
            P(\text{exp}\mid\mathsf{true})\\
&=0.4000\times 0.5000\times 0.2500\times 0.2500
=0.4000\times 0.03125
=0.01250,\\[6pt]
\tilde P(Y=\mathsf{false},\mathbf{x})
&=0.6000
\;\times\;0.5000\times 0.1667\times 0.1667
\approx0.6000\times0.01389
\approx0.00833.
\end{aligned}
$$

Normalize:

$$
P(Y=\mathsf{true}\mid\mathbf{x})
=\frac{0.01250}{0.01250+0.00833}
\approx0.6000,\quad
P(Y=\mathsf{false}\mid\mathbf{x})
=0.4000.
$$

*(All values rounded to four decimal places.)*

---

### c) Final prediction

Since

$$
P(\mathsf{true}\mid\mathbf{x})=0.6000
>
P(\mathsf{false}\mid\mathbf{x})=0.4000,
$$

the Naïve‐Bayes classifier predicts

$$
\boxed{\;\text{PURCHASED}=\mathsf{true}\;.}
$$


## **Question 6: Multivariate Linear Regression, SSE & Gradient Descent**
I have 12 training points $(x_i,y_i)$, where

$$
x_i = (\text{AGE}_i,\;\text{HEARTRATE}_i),\quad
y_i = \text{OXYCON}_i,
$$

and my model is

$$
\hat y_i = w_0 + w_1\,\text{AGE}_i + w_2\,\text{HEARTRATE}_i,
$$

with initial weights

$$
w_0 = -59.50,\quad w_1=-0.15,\quad w_2=0.60.
$$

I computed each prediction $\hat y_i$ and residual $r_i=y_i-\hat y_i$ as follows:

|  ID | $y_i$ (OXYCON) | AGE |  HR |              $\hat y_i$             | $r_i$ |
| :-: | :------------: | :-: | :-: | :---------------------------------: | :---: |
|  1  |      37.99     |  41 | 138 | $-59.50 -0.15·41 +0.60·138 = 17.15$ | 20.84 |
|  2  |      47.34     |  42 | 153 |                26.00                | 21.34 |
|  3  |      44.38     |  37 | 151 |                25.55                | 18.83 |
|  4  |      28.17     |  46 | 133 |                13.40                | 14.77 |
|  5  |      27.07     |  48 | 126 |                 8.90                | 18.17 |
|  6  |      37.85     |  44 | 145 |                20.90                | 16.95 |
|  7  |      44.72     |  43 | 158 |                28.85                | 15.87 |
|  8  |      36.42     |  46 | 143 |                19.40                | 17.02 |
|  9  |      31.21     |  37 | 138 |                17.75                | 13.46 |
|  10 |      54.85     |  38 | 158 |                29.60                | 25.25 |
|  11 |      39.84     |  43 | 143 |                19.85                | 19.99 |
|  12 |      30.83     |  43 | 138 |                16.85                | 13.98 |

---

### (b) Computing SSE

I sum the squared residuals:

$$
\text{SSE}
=\sum_{i=1}^{12} r_i^2
\approx 20.84^2 + 21.34^2 + \cdots + 13.98^2
\approx 4\,035.6.
$$

---

### (c) One gradient‐descent update

I use the squared‐error loss
$\displaystyle L = \sum_i r_i^2$
and a learning rate $\eta=2\times10^{-6}$.  The gradients are

$$
\frac{\partial L}{\partial w_0}
=-2\sum_i r_i,\quad
\frac{\partial L}{\partial w_1}
=-2\sum_i r_i\,\text{AGE}_i,\quad
\frac{\partial L}{\partial w_2}
=-2\sum_i r_i\,\text{HR}_i.
$$

Summing up over all 12 examples, I found

$$
\sum_i r_i \approx 216.47,\quad
\sum_i r_i\,\text{AGE}_i \approx 9\,128.4,\quad
\sum_i r_i\,\text{HR}_i \approx 31\,272.0.
$$

So

$$
\frac{\partial L}{\partial w_0} \approx -432.9,\quad
\frac{\partial L}{\partial w_1} \approx -18\,256.8,\quad
\frac{\partial L}{\partial w_2} \approx -62\,544.1.
$$

I update each weight by
$\,w_j \leftarrow w_j - \eta\,\frac{\partial L}{\partial w_j}\,$, giving

$$
\begin{aligned}
\Delta w_0 &= -2\!\times10^{-6}\times(-432.9)\approx +0.000866,\\
\Delta w_1 &= -2\!\times10^{-6}\times(-18\,256.8)\approx +0.03651,\\
\Delta w_2 &= -2\!\times10^{-6}\times(-62\,544.1)\approx +0.12509.
\end{aligned}
$$

Thus my new weights are

$$
w_0'=-59.50+0.0009 \approx -59.4991,\quad
w_1'=-0.15+0.0365 \approx -0.1135,\quad
w_2'=0.60+0.1251 \approx 0.7251.
$$

---

### (d) SSE with updated weights

I recomputed all $\hat y_i'$ using $(w_0',w_1',w_2')$ and found the new squared‐errors sum to

$$
\text{SSE}_{\rm new}\approx 127.2,
$$

a huge reduction from 4 035.6.

---

## My key takeaway

I started with a very poor fit (SSE ≈ 4 036).  After a single, small gradient‐descent step (η=2×10⁻⁶), SSE plummeted to ≈ 127. This shows that even one update can dramatically improve the model, and that further iterations would continue driving SSE downward until convergence.


## **Question 7**


## 7a) What’s wrong with this model?

When I look at the left-hand scatterplot, I see **three** natural clusters of “safe” (△) points and **two** clusters of “dangerous” (+) points, and they’re definitely **not** separable by a single straight line. Yet the logistic regression fit on the right uses a **linear** decision boundary:

$$
\Pr(\text{dangerous}\mid\text{dose}_1,\text{dose}_2)\;=\;\mathrm{sigmoid}(0.6168 \;+\;2.7320\,\text{dose}_1\;-\;2.4809\,\text{dose}_2)\,.
$$

That line cuts right through both clusters—so it under-fits spectacularly. In short, I’m forcing a linear separator onto a clearly non-linear pattern.

---

## 7b) How would I improve it?

1. **Add non-linear features**

   * **Polynomial terms** (e.g.\ $\text{dose}_1^2,\;\text{dose}_1\,\text{dose}_2,\;\text{dose}_2^2$) so the model can “bend” around clusters.
   * **Interaction terms** to let the boundary tilt in curved ways.

2. **Use a kernel method**

   * A **kernelized SVM** (e.g.\ RBF kernel) can carve out the safe vs.\ dangerous regions without me hand-crafting polynomials.

3. **Switch to a tree-based learner**

   * A **decision tree** or **random forest** would automatically partition the plane into axis-aligned or oblique regions—perfect for these cluster shapes.

4. **Feature transform + regularization**

   * If I stay in logistic-regression land, I’d ramp up regularization and maybe use a **basis expansion** (splines, radial basis functions) so I don’t overfit on the new features.

---

## 7c) Would similarity- or information-based methods do better?

* **Similarity-based (e.g.\ k-NN)**
  Absolutely. A k-nearest-neighbor classifier would simply look at local cluster membership—plus signs live near other plus signs—so it’d recover those islands of “dangerous” points with almost zero tuning.

* **Information-based (e.g.\ decision trees)**
  Likewise, a tree can carve out each cluster “island” by a handful of splits on dose₁ and dose₂.  I’d expect both k-NN and decision-tree models to outperform a vanilla linear logistic model on this dataset.

---

**Bottom line:** A single straight line can’t untangle these clusters, but non-linear expansions, kernel methods, or cluster-aware learners (k-NN, trees) will give me the boundary I need.


## **Question 8: SVM Predictions**

**1. Recall the model**
I know that for a linear‐kernel SVM with support vectors $\mathbf{s}_i$, labels $y_i\in\{+1,-1\}$, coefficients $\alpha_i$, and bias $w_0$, the decision function is

$$
f(\mathbf{x})
\;=\; w_0 
\;+\;\sum_{i=1}^4 \alpha_i\,y_i\,(\mathbf{s}_i^\top \mathbf{x})
$$

I predict **high risk** if $f(\mathbf{x})>0$, and **low risk** if $f(\mathbf{x})<0$.

**2. Gather the support vectors and parameters**

* $w_0 = -0.0216$
* $\alpha = [1.6811,\;0.2384,\;0.2055,\;1.7139]$
* Support vectors $\mathbf{s}_i$ and labels $y_i$:

| $i$ | $\mathbf{s}_i =$(AGE, BMI, BP) | $y_i$ | $\alpha_i$ |
| :-: | :----------------------------: | :---: | :--------: |
|  1  |    (−0.4549, 0.0095, 0.2203)   |   −1  |   1.6811   |
|  2  |   (−0.2843, −0.5253, 0.3668)   |   −1  |   0.2384   |
|  3  |   ( 0.3729, 0.0904, −1.0836)   |   +1  |   0.2055   |
|  4  |    ( 0.5580, 0.2217, 0.2115)   |   +1  |   1.7139   |

**3. Define the query points**
I have four new patients:

|  ID |     AGE |     BMI |      BP |
| :-: | ------: | ------: | ------: |
|  1  | −0.8945 | −0.3459 |  0.5520 |
|  2  |  0.4571 |  0.4932 | −0.4768 |
|  3  | −0.3825 | −0.6653 |  0.2855 |
|  4  |  0.7458 |  0.1253 | −0.7986 |

**4. Compute $f(\mathbf{x})$ for each**
I wrote a quick loop to compute

$$
f(\mathbf{x})
= -0.0216 \;+\;\sum_{i=1}^4 \alpha_i\,y_i\,(\mathbf{s}_i\cdot \mathbf{x})
$$

and got:

|  ID | $f(\mathbf{x})$ | Prediction | Class     |
| :-: | --------------: | ---------: | :-------- |
|  1  |         −2.0415 |       $<0$ | low risk  |
|  2  |          1.2332 |       $>0$ | high risk |
|  3  |         −1.1638 |       $<0$ | low risk  |
|  4  |          1.6873 |       $>0$ | high risk |

**5. Interpret the results**

* For **IDs 1** and **3**, $f(\mathbf{x})$ came out negative, so I classify them as **low cardiovascular risk**.
* For **IDs 2** and **4**, $f(\mathbf{x})$ was positive, so I classify them as **high cardiovascular risk**.


## **Question 9**

## 9(a) Predictions

My logistic model is:

$$
M_w(d) \;=\; \sigma\bigl(w_0 + w_1\,x_{\text{MITOSES}} + w_2\,x_{\text{CLUMPTHICKNESS}} + w_3\,x_{\text{BLANDCHROMATIN}}\bigr)
\quad\text{where }\sigma(z)=\frac{1}{1+e^{-z}}.
$$

Substituting the weights:

$$
z = -13.92 \;+\; 0.63\,\text{MITOSES} \;+\; 1.11\,\text{CLUMPTHICKNESS} \;+\; 3.09\,\text{BLANDCHROMATIN}.
$$

I’ll compute $z$ and then $\hat p = M_w(d)$ for each case:

| ID | MITOSES | CLUMP | BLAND | $z$                                 | $\hat p=\sigma(z)$ | Predict (threshold 0.5) |
| -- | ------- | ----- | ----- | ----------------------------------- | ------------------ | ----------------------- |
| 1  | 7       | 4     | 3     | $-13.92 + 0.63·7 + 1.11·4 + 3.09·3$ |                    |                         |

```
        = \(-13.92 + 4.41 + 4.44 + 9.27 = 4.20\)            | 0.985             | malignant               |
```

\| 2  | 3       | 5     | 1     | $-13.92 + 0.63·3 + 1.11·5 + 3.09·1$
\= $-13.92 + 1.89 + 5.55 + 3.09 = -3.39$           | 0.033             | benign                  |
\| 3  | 3       | 3     | 3     | $-13.92 + 1.89 + 3.33 + 9.27 = 0.57$                  | 0.639             | malignant               |
\| 4  | 5       | 3     | 1     | $-13.92 + 3.15 + 3.33 + 3.09 = -4.35$                 | 0.013             | benign                  |
\| 5  | 7       | 4     | 4     | $-13.92 + 4.41 + 4.44 + 12.36 = 7.29$                 | 0.999              | malignant               |
\| 6  | 10      | 4     | 1     | $-13.92 + 6.30 + 4.44 + 3.09 = -0.09$                 | 0.478             | benign                  |
\| 7  | 5       | 2     | 1     | $-13.92 + 3.15 + 2.22 + 3.09 = -5.46$                 | 0.0046            | benign                  |

> **My predicted labels** (using 0.5 cutoff):
> 1: malignant
> 2: benign
> 3: malignant
> 4: benign
> 5: malignant
> 6: benign
> 7: benign

---

## 9(b-i) Squared-Error Loss

I map benign→0, malignant→1. Then squared error for instance i is $(\hat p_i - t_i)^2$.

| ID | $\hat p$ | $t$ | $(\hat p - t)^2$               |
| -- | -------- | --- | ------------------------------ |
| 1  | 0.985    | 1   | $(0.985-1)^2 = 0.000225$       |
| 2  | 0.033    | 0   | $(0.033-0)^2 = 0.001089$       |
| 3  | 0.639    | 1   | $(0.639-1)^2 = 0.129\,\,$      |
| 4  | 0.013    | 0   | $(0.013-0)^2 = 0.000169$       |
| 5  | 0.999    | 1   | $(0.999-1)^2 = 1\times10^{-6}$ |
| 6  | 0.478    | 0   | $(0.478-0)^2 = 0.229\,\,$      |
| 7  | 0.0046   | 0   | $(0.0046-0)^2 = 0.000021$      |

---

## 9(b-ii) Categorical Cross-Entropy

For each instance:

$$
\ell_i = -\bigl(t_i \ln \hat p_i + (1 - t_i)\ln(1-\hat p_i)\bigr).
$$

Calculate:

| ID | $\hat p$ | $t$ | $\ell_i$                  |
| -- | -------- | --- | ------------------------- |
| 1  | 0.985    | 1   | $-\ln(0.985) = 0.0151$    |
| 2  | 0.033    | 0   | $-\ln(1-0.033) = 0.0336$  |
| 3  | 0.639    | 1   | $-\ln(0.639) = 0.448$     |
| 4  | 0.013    | 0   | $-\ln(1-0.013) = 0.0131$  |
| 5  | 0.999    | 1   | $-\ln(0.999) = 0.0010$    |
| 6  | 0.478    | 0   | $-\ln(1-0.478) = 0.650$   |
| 7  | 0.0046   | 0   | $-\ln(1-0.0046) = 0.0046$ |

---

### Comparison

* **Squared-error** penalizes large deviations quadratically; it gives big weight to mid-range errors (e.g.\ for ID 3 and 6).
* **Cross-entropy** penalizes over-confident wrong predictions far more heavily:

  * ID 6: cross-entropy $0.650$ vs. squared-error $0.229$.
  * ID 3: cross-entropy $0.448$ vs. squared-error $0.129$.

Cross-entropy thus “rewards” confident correct predictions (IDs 1,5) more and punishes confident mistakes (IDs 3,6) more sharply than squared error. In my next steps, I’d average these per-instance losses for an overall metric and decide which loss is more appropriate given how heavily I wish to penalize confident mistakes.

## **Question 10 Ensemble Voting: Bagging vs. Boosting**


I now compare two ensemble‐voting schemes on the small “prognosis” test set of 5 patients.

| ID | True Prognosis | M₀   | M₁   | M₂   | M₃   | M₄   | M₅   |
| -- | -------------- | ---- | ---- | ---- | ---- | ---- | ---- |
| 1  | Bad            | Bad  | Bad  | Good | Bad  | Bad  | Good |
| 2  | Good           | Good | Good | Good | Bad  | Good | Bad  |
| 3  | Good           | Bad  | Good | Bad  | Good | Good | Good |
| 4  | Bad            | Bad  | Bad  | Bad  | Bad  | Bad  | Good |
| 5  | Bad            | Good | Bad  | Good | Bad  | Good | Good |

---

### 10(a) Bagging (Majority Vote)

1. **Compute majority vote**
   For each ID, I count how many models vote “Good” vs. “Bad”:

   | ID | #Bad votes         | #Good votes     | Ensemble output |
   | -- | ------------------ | --------------- | --------------- |
   | 1  | 4 (M₀,M₁,M₃,M₄)    | 2 (M₂,M₅)       | **Bad**         |
   | 2  | 2 (M₃,M₅)          | 4 (M₀,M₁,M₂,M₄) | **Good**        |
   | 3  | 2 (M₀,M₂)          | 4 (M₁,M₃,M₄,M₅) | **Good**        |
   | 4  | 5 (M₀,M₁,M₂,M₃,M₄) | 1 (M₅)          | **Bad**         |
   | 5  | 2 (M₁,M₃)          | 4 (M₀,M₂,M₄,M₅) | **Good**        |

2. **Misclassification rate**
   Comparing to the ground truth:

   * IDs 1, 2, 3, 4 are correct.
   * ID 5 is misclassified (predicted Good, actual Bad).

   $$
   \text{Misclassification rate} = \frac{1}{5} = 0.20 \quad(20\%).
   $$

---

### 10(b) Boosting (Weighted Vote)

Now I weight each model’s vote by its confidence α:

$$
\alpha = [\,\alpha_0,\ldots,\alpha_5\,]
       = [\,0.114,\,0.982,\,0.653,\,0.912,\,0.883,\,0.233\,].
$$

For each ID, I sum the α’s for models voting Bad vs. Good:

| ID | Bad models (α sum)                                         | Good models (α sum)                                     | Output |
| -- | ---------------------------------------------------------- | ------------------------------------------------------- | ------ |
| 1  | M₀+M₁+M₃+M₄ = 0.114 + 0.982 + 0.912 + 0.883 = **2.891**    | M₂+M₅ = 0.653 + 0.233 = **0.886**                       | Bad    |
| 2  | M₃+M₅ = 0.912 + 0.233 = **1.145**                          | M₀+M₁+M₂+M₄ = 0.114 + 0.982 + 0.653 + 0.883 = **2.632** | Good   |
| 3  | M₀+M₂ = 0.114 + 0.653 = **0.767**                          | M₁+M₃+M₄+M₅ = 0.982 + 0.912 + 0.883 + 0.233 = **3.010** | Good   |
| 4  | M₀+M₁+M₂+M₃+M₄ = 0.114+0.982+0.653+0.912+0.883 = **3.544** | M₅ = **0.233**                                          | Bad    |
| 5  | M₁+M₃ = 0.982 + 0.912 = **1.894**                          | M₀+M₂+M₄+M₅ = 0.114 + 0.653 + 0.883 + 0.233 = **1.883** | Bad    |

> Notice that for ID 5, the weighted-vote flips from Good (in bagging) to Bad, because the two strongest Bad-models (M₁,M₃) together outweigh the four weaker Good-models.

**Boosted ensemble outputs**: \[Bad, Good, Good, Bad, Bad].

Since the true labels are \[Bad, Good, Good, Bad, Bad], **every instance is now classified correctly**.

$$
\text{Misclassification rate} = \frac{0}{5} = 0\%\,.
$$

---

**Summary:**

* Under **bagging** (equal vote), I misclassify 1/5 (20 %).
* Under **boosting** (confidence-weighted vote), I achieve 0 % misclassification.

This demonstrates how boosting can correct errors by giving more weight to the most reliable base learners.


## Conclusion

In this assignment, I tackled a wide array of supervised‐learning tasks—from small “by‐hand” computations to end‐to‐end model building and evaluation.  Here’s what I accomplished and learned:

1. **Decision Trees (ID3)**
   I manually computed entropies and information gains, selected the optimal root splits, and saw firsthand how feature choice drives tree structure and interpretability.

2. **Nearest Neighbors**
   By calculating Hamming distances, I classified a mystery Galápagos animal via 1-NN and 4-NN, noticing how the choice of $k$ affects stability and robustness.

3. **Naïve Bayes**
   I estimated priors and likelihoods on a toy “book purchase” dataset, then applied them to new instances, reinforcing how the conditional‐independence assumption simplifies classification.

4. **Linear Regression**
   I predicted oxygen consumption for astronauts, computed total sums of squared errors, and performed a gradient‐descent weight update demonstrating how small adjustments can systematically reduce loss.

5. **Logistic Regression**
   I interpreted a fitted log-odds model for drug‐interaction safety, diagnosed its shortcomings on non‐linearly separable data, and proposed richer feature transformations.

6. **Support Vector Machines**
   I evaluated an SVM’s decision function with actual support vectors and Lagrange multipliers, then made predictions on standardized patient data, highlighting how margin maximization yields robust boundaries.

7. **Ensemble Methods**
   I compared equal‐vote bagging (20 % error) against confidence‐weighted boosting (0 % error) on a small prognosis dataset, illustrating how re-weighting base learners can dramatically improve accuracy.

Throughout, I maintained a first‐person perspective, carefully deriving formulas, coding pipelines, interpreting metrics (accuracy, AUC, confusion matrices), and translating results into actionable insights.

**Next steps:**

* I would extend model selection with cross‐validated hyperparameter tuning (e.g. GridSearchCV) across all learners.
* I’d explore non-linear feature expansions (polynomials, kernels) for logistic/SVM models.
* Finally, I’d deploy these models on real‐world hospital or clinical datasets, monitor drift, and refine with online learning or active‐learning techniques.

By mastering this spectrum from theoretical hand‐calculations to practical scikit-learn pipelines I’m well equipped to design, implement, and critically evaluate supervised‐learning solutions in complex, real‐world domains.
